In [9]:
import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from pathlib import Path

In [10]:
from utils import score, compute_costs, objective

In [11]:
x_train = pd.read_csv(Path.cwd().parent / "data/german_credit_train.csv")
x_test = pd.read_csv(Path.cwd().parent / "data/german_credit_test.csv")

In [12]:
x_test.drop(columns='Id')


,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
0,no_checking,9,prior_payments_delayed,car_new,1032,100_to_500,4_to_7,3,male,none,4,savings_insurance,41,none,own,1,management_self-employed,1,none,yes
1,less_0,5,all_credits_paid_back,car_new,1523,less_100,unemployed,2,female,none,2,real_estate,19,none,rent,1,management_self-employed,1,none,yes
2,no_checking,39,prior_payments_delayed,repairs,7150,500_to_1000,4_to_7,3,male,co-applicant,4,unknown,52,none,own,2,skilled,1,yes,yes
3,0_to_200,15,prior_payments_delayed,furniture,250,500_to_1000,4_to_7,3,male,none,2,savings_insurance,24,none,own,2,skilled,2,yes,yes
4,0_to_200,16,prior_payments_delayed,car_new,5551,100_to_500,1_to_4,3,male,none,3,car_other,34,none,rent,2,management_self-employed,1,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,no_checking,38,credits_paid_to_date,appliances,5308,100_to_500,4_to_7,3,male,none,3,savings_insurance,31,none,own,1,skilled,1,none,yes
997,less_0,31,credits_paid_to_date,retraining,1997,less_100,1_to_4,3,male,none,3,savings_insurance,31,none,own,2,skilled,2,none,yes
998,less_0,20,prior_payments_delayed,radio_tv,1155,greater_1000,1_to_4,3,male,none,3,savings_insurance,33,none,rent,2,skilled,1,yes,yes
999,less_0,4,credits_paid_to_date,car_new,250,less_100,unemployed,1,female,none,1,real_estate,23,none,rent,1,skilled,1,none,yes


In [13]:
y_train = x_train[['LoanAmount', 'Risk']]
x_train = x_train.drop(columns='Risk')

In [14]:
X_valid, X_predictions, Y_valid, Y_predictions = train_test_split(x_train, y_train)

In [15]:
Y_valid = Y_valid.drop(columns = 'LoanAmount') # Otherwise the model won't know what to aim for

In [16]:
study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective(trial, X_valid, Y_valid, X_predictions, Y_predictions), n_trials=100, timeout=600, n_jobs=-1)

study.trials_dataframe()
print(f"Best model reach a validation score of: {study.best_value} and is reached for a learning rate of: {study.best_params}")

[I 2025-03-05 13:40:45,451] A new study created in memory with name: no-name-7b175bca-7701-4ff8-9046-eb0046a819e8


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

Best model reach a validation score of: -98.30075287356318 and is reached for a learning rate of: {'learning_rate': 0.0014871668687017892, 'max_iter': 153, 'max_leaf_nodes': 89, 'min_samples_leaf': 37, 'l2_regularization': 6.568566293109978, 'validation_fraction': 0.21143887620628912, 'n_iter_no_change': 12}
